# Modelado y comparación de supervivencia — METABRIC

Este notebook continúa el trabajo de EDA y preprocesamiento ya realizado sobre **METABRIC** y carga directamente los artefactos generados al final del notebook anterior:

- `X_train_df.parquet`, `X_test_df.parquet`
- `X_train_np.npy`, `X_test_np.npy`
- `dur_train.npy`, `dur_test.npy`
- `evt_train.npy`, `evt_test.npy`
- `y_train.pkl`, `y_test.pkl` *(opcional; se reconstruye desde duración/evento para evitar problemas de compatibilidad)*

Se implementan y comparan cuatro enfoques de análisis de supervivencia:

1. **Kaplan-Meier estratificado** como baseline interpretable.
2. **Cox penalizado LASSO** (`CoxnetSurvivalAnalysis`, penalización L1).
3. **Random Survival Forest**.
4. **DeepSurv** mediante `pycox`/`PyTorch`.

Las métricas principales son:

- **C-index de Harrell**: mayor es mejor.
- **C-index IPCW/Uno**: mayor es mejor, corrige parcialmente la censura.
- **Integrated Brier Score (IBS)**: menor es mejor.
- **Log-rank test** entre grupos de alto y bajo riesgo: p-valor bajo indica separación significativa de curvas.

## 0. Requisitos y configuración de ejecución

Antes de ejecutar este notebook:

1. Ejecuta el notebook de EDA/preprocesamiento hasta la celda de guardado de `../data/processed/metabric`.
2. Coloca este notebook dentro de la carpeta `notebooks/` del proyecto, o ajusta `DATA_DIR`.
3. Comprueba que las dependencias están instaladas.

> Si usas un entorno nuevo, puedes descomentar la celda siguiente. En algunos sistemas `scikit-survival` puede requerir una versión reciente de Python y compiladores adecuados.

In [ ]:
# Descomentar solo si faltan dependencias en tu entorno.
# %pip install -q lifelines scikit-survival pycox torchtuples torch pyarrow joblib pandas numpy scikit-learn matplotlib seaborn

In [ ]:
# ===============================
# Parámetros globales del notebook
# ===============================

from pathlib import Path

RANDOM_STATE = 42

# Ruta esperada si este notebook se ejecuta desde /notebooks
DATA_DIR = Path("../data/processed/metabric")

# Rutas alternativas por si se ejecuta desde la raíz del proyecto
ALT_DATA_DIRS = [
    DATA_DIR,
    Path("data/processed/metabric"),
    Path.cwd() / "../data/processed/metabric",
    Path.cwd() / "data/processed/metabric",
]

MODEL_DIR = Path("../models/metabric_survival")
REPORT_DIR = Path("../reports/metabric_survival")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Evaluación
N_EVAL_TIMES = 100
TEST_SIZE_DEEPSURV_VAL = 0.20

# Cox LASSO
COX_N_ALPHAS = 60
COX_INNER_CV_SPLITS = 5

# RSF
RSF_N_ESTIMATORS = 500
RSF_MIN_SAMPLES_SPLIT = 10
RSF_MIN_SAMPLES_LEAF = 15
RSF_MAX_FEATURES = "sqrt"

# DeepSurv
DEEPSURV_NODES = [64, 32]
DEEPSURV_DROPOUT = 0.20
DEEPSURV_BATCH_SIZE = 128
DEEPSURV_EPOCHS = 256
DEEPSURV_PATIENCE = 25
DEEPSURV_LR = 1e-3

# Validación cruzada global opcional.
# Recomendación: dejar en False durante pruebas rápidas y poner en True para resultados finales.
RUN_INTERNAL_CV = False
CV_SPLITS = 5
CV_DEEPSURV_EPOCHS = 96

In [ ]:
# =================
# Imports generales
# =================

import os
import random
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.inspection import permutation_importance

from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

from sksurv.util import Surv
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import (
    concordance_index_censored,
    concordance_index_ipcw,
    integrated_brier_score,
)

import torch
import torchtuples as tt
from pycox.models import CoxPH as DeepSurvModel

warnings.filterwarnings("ignore")

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

print("✓ Imports completados")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")

## 1. Carga de los datos procesados

La carga se hace desde los ficheros guardados por el notebook de preprocesamiento.  
Si el notebook no encuentra los ficheros, se detiene con un mensaje explícito para evitar ejecutar modelos sobre datos incorrectos.

In [ ]:
# ===================================
# Localización robusta de DATA_DIR
# ===================================

required_files = [
    "X_train_np.npy",
    "X_test_np.npy",
    "dur_train.npy",
    "dur_test.npy",
    "evt_train.npy",
    "evt_test.npy",
]

resolved_data_dir = None
for candidate in ALT_DATA_DIRS:
    if all((candidate / f).exists() for f in required_files):
        resolved_data_dir = candidate
        break

if resolved_data_dir is None:
    checked = "\n".join([f"  - {p.resolve()}" for p in ALT_DATA_DIRS])
    raise FileNotFoundError(
        "No se han encontrado los artefactos procesados de METABRIC.\n"
        "Ejecuta primero el notebook de EDA/preprocesamiento hasta la celda de guardado.\n"
        f"Rutas comprobadas:\n{checked}"
    )

DATA_DIR = resolved_data_dir
print(f"✓ Datos encontrados en: {DATA_DIR.resolve()}")

In [ ]:
# ===============================
# Carga de X, duración y eventos
# ===============================

# Intento preferente: DataFrames parquet con nombres de columnas
try:
    X_train = pd.read_parquet(DATA_DIR / "X_train_df.parquet")
    X_test = pd.read_parquet(DATA_DIR / "X_test_df.parquet")
    print("✓ X cargada desde parquet con nombres de covariables.")
except Exception as exc:
    print("⚠️ No se pudieron cargar los parquet. Se usarán matrices numpy con nombres genéricos.")
    print(f"Motivo: {exc}")
    X_train_np_tmp = np.load(DATA_DIR / "X_train_np.npy")
    X_test_np_tmp = np.load(DATA_DIR / "X_test_np.npy")
    feature_names_tmp = [f"feature_{i:03d}" for i in range(X_train_np_tmp.shape[1])]
    X_train = pd.DataFrame(X_train_np_tmp, columns=feature_names_tmp)
    X_test = pd.DataFrame(X_test_np_tmp, columns=feature_names_tmp)

# Matrices numpy para modelos que las requieren
X_train_np = X_train.to_numpy(dtype=np.float32)
X_test_np = X_test.to_numpy(dtype=np.float32)
feature_names = X_train.columns.to_list()

# Duración y evento
dur_train = np.load(DATA_DIR / "dur_train.npy").astype(float)
dur_test = np.load(DATA_DIR / "dur_test.npy").astype(float)
evt_train = np.load(DATA_DIR / "evt_train.npy").astype(bool)
evt_test = np.load(DATA_DIR / "evt_test.npy").astype(bool)

# Structured arrays para scikit-survival
y_train = Surv.from_arrays(event=evt_train, time=dur_train)
y_test = Surv.from_arrays(event=evt_test, time=dur_test)

# Validaciones básicas
assert len(X_train) == len(dur_train) == len(evt_train), "Inconsistencia en train."
assert len(X_test) == len(dur_test) == len(evt_test), "Inconsistencia en test."
assert X_train.shape[1] == X_test.shape[1], "Train y test no tienen el mismo número de columnas."

summary = pd.DataFrame({
    "partición": ["train", "test"],
    "n": [len(X_train), len(X_test)],
    "n_eventos": [int(evt_train.sum()), int(evt_test.sum())],
    "tasa_eventos": [evt_train.mean(), evt_test.mean()],
    "duración_mediana_meses": [np.median(dur_train), np.median(dur_test)],
    "duración_max_meses": [np.max(dur_train), np.max(dur_test)],
    "n_covariables": [X_train.shape[1], X_test.shape[1]],
})

display(summary)
print(f"Número de covariables: {len(feature_names)}")

## 2. Funciones auxiliares de evaluación

Estas funciones centralizan la evaluación para que los cuatro modelos se comparen bajo el mismo criterio.

Notas metodológicas:

- El **C-index de Harrell** evalúa discriminación por ranking.
- El **C-index IPCW** es útil cuando hay censura relevante.
- El **IBS** requiere probabilidades de supervivencia estimadas en una malla temporal común.
- El **log-rank** se calcula dividiendo el riesgo en dos grupos: bajo y alto riesgo, usando la mediana del score en test.

In [ ]:
# ==========================================
# Funciones auxiliares para supervivencia
# ==========================================

EPS_TIME = 1e-6

def make_eval_times(y_train, y_test, n_times=100):
    """Genera tiempos de evaluación válidos para IBS evitando extremos problemáticos."""
    t_train = y_train["time"].astype(float)
    t_test = y_test["time"].astype(float)

    # Tau común para evitar extrapolación fuera del rango de entrenamiento/test.
    tau = min(np.max(t_train), np.max(t_test)) - EPS_TIME

    lower = max(
        np.percentile(t_test, 5),
        np.percentile(t_train, 5),
        np.min(t_test) + EPS_TIME,
        np.min(t_train) + EPS_TIME,
    )
    upper = min(
        np.percentile(t_test, 95),
        np.percentile(t_train, 95),
        tau,
    )

    if not np.isfinite(lower) or not np.isfinite(upper) or lower >= upper:
        lower = max(np.min(t_test), np.min(t_train)) + EPS_TIME
        upper = tau

    if lower >= upper:
        raise ValueError("No se pudo construir una malla temporal válida para IBS.")

    return np.linspace(lower, upper, n_times)


def harrell_cindex(y_true, risk_scores):
    """C-index de Harrell. risk_scores altos = mayor riesgo."""
    return float(
        concordance_index_censored(
            y_true["event"].astype(bool),
            y_true["time"].astype(float),
            np.asarray(risk_scores, dtype=float)
        )[0]
    )


def uno_cindex(y_train_ref, y_true, risk_scores):
    """C-index IPCW/Uno. Devuelve NaN si la estimación no es posible."""
    try:
        tau = min(np.max(y_train_ref["time"]), np.max(y_true["time"])) - EPS_TIME
        return float(concordance_index_ipcw(y_train_ref, y_true, risk_scores, tau=tau)[0])
    except Exception as exc:
        print(f"⚠️ C-index IPCW no calculable: {exc}")
        return np.nan


def safe_integrated_brier_score(y_train_ref, y_true, survival_estimates, eval_times):
    """IBS con control de errores y clipping de probabilidades."""
    try:
        survival_estimates = np.asarray(survival_estimates, dtype=float)
        survival_estimates = np.nan_to_num(survival_estimates, nan=1.0, posinf=1.0, neginf=0.0)
        survival_estimates = np.clip(survival_estimates, 0.0, 1.0)
        return float(integrated_brier_score(y_train_ref, y_true, survival_estimates, eval_times))
    except Exception as exc:
        print(f"⚠️ IBS no calculable: {exc}")
        return np.nan


def logrank_from_risk(durations, events, risk_scores, threshold=None):
    """Log-rank entre bajo/alto riesgo a partir del score predicho."""
    risk_scores = np.asarray(risk_scores, dtype=float)

    if np.allclose(risk_scores, risk_scores[0]):
        return np.nan, np.nan, None

    if threshold is None:
        threshold = np.median(risk_scores)

    high_risk = risk_scores >= threshold

    # Evita grupos vacíos por empates exactos en la mediana
    if high_risk.sum() == 0 or (~high_risk).sum() == 0:
        threshold = np.quantile(risk_scores, 0.60)
        high_risk = risk_scores >= threshold

    if high_risk.sum() == 0 or (~high_risk).sum() == 0:
        return np.nan, np.nan, None

    result = logrank_test(
        durations_A=durations[high_risk],
        durations_B=durations[~high_risk],
        event_observed_A=events[high_risk],
        event_observed_B=events[~high_risk],
    )
    return float(result.test_statistic), float(result.p_value), high_risk


def eval_step_functions(surv_funcs, times):
    """Evalúa una lista de StepFunction de scikit-survival en la malla temporal común."""
    estimates = []
    for fn in surv_funcs:
        # StepFunction de sksurv suele exponer x/y.
        if hasattr(fn, "x"):
            x_min = float(np.min(fn.x))
            x_max = float(np.max(fn.x))
            clipped_times = np.clip(times, x_min, x_max)
            vals = fn(clipped_times)
            vals = np.where(times < x_min, 1.0, vals)
        else:
            vals = fn(times)

        estimates.append(np.asarray(vals, dtype=float))

    estimates = np.vstack(estimates)
    return np.clip(np.nan_to_num(estimates, nan=1.0), 0.0, 1.0)


def pycox_surv_df_to_array(surv_df, times):
    """
    Convierte el DataFrame de supervivencia de pycox:
    index = tiempos, columnas = pacientes
    en array shape = (n_pacientes, n_tiempos).
    Se usa evaluación escalonada: último valor conocido antes de cada tiempo.
    """
    time_grid = surv_df.index.to_numpy(dtype=float)
    values = surv_df.to_numpy(dtype=float)  # shape: n_times_model x n_patients

    idx = np.searchsorted(time_grid, times, side="right") - 1
    estimates = np.empty((values.shape[1], len(times)), dtype=float)

    for j, pos in enumerate(idx):
        if pos < 0:
            estimates[:, j] = 1.0
        else:
            estimates[:, j] = values[pos, :]

    return np.clip(np.nan_to_num(estimates, nan=1.0), 0.0, 1.0)


def register_result(results, model_name, risk_scores, survival_estimates, eval_times, notes=""):
    """Calcula todas las métricas principales y añade una fila a results."""
    c_h = harrell_cindex(y_test, risk_scores)
    c_u = uno_cindex(y_train, y_test, risk_scores)
    ibs = safe_integrated_brier_score(y_train, y_test, survival_estimates, eval_times)
    lr_stat, lr_p, high_risk = logrank_from_risk(dur_test, evt_test, risk_scores)

    results.append({
        "modelo": model_name,
        "c_index_harrell_test": c_h,
        "c_index_ipcw_test": c_u,
        "ibs_test": ibs,
        "logrank_stat_test": lr_stat,
        "logrank_p_test": lr_p,
        "n_alto_riesgo_test": int(high_risk.sum()) if high_risk is not None else np.nan,
        "n_bajo_riesgo_test": int((~high_risk).sum()) if high_risk is not None else np.nan,
        "notas": notes,
    })

    return high_risk


def plot_km_by_group(durations, events, group_mask, title):
    """Curva Kaplan-Meier para grupos bajo/alto riesgo en test."""
    if group_mask is None:
        print(f"No se puede representar {title}: grupos de riesgo no definidos.")
        return

    kmf_low = KaplanMeierFitter()
    kmf_high = KaplanMeierFitter()

    plt.figure(figsize=(7, 5))
    kmf_low.fit(
        durations[~group_mask],
        event_observed=events[~group_mask],
        label="Bajo riesgo"
    )
    kmf_high.fit(
        durations[group_mask],
        event_observed=events[group_mask],
        label="Alto riesgo"
    )
    ax = kmf_low.plot(ci_show=True)
    kmf_high.plot(ax=ax, ci_show=True)
    plt.title(title)
    plt.xlabel("Tiempo de supervivencia (meses)")
    plt.ylabel("Probabilidad de supervivencia")
    plt.grid(alpha=0.25)
    plt.show()


eval_times = make_eval_times(y_train, y_test, n_times=N_EVAL_TIMES)
print(f"Malla temporal IBS: {eval_times[0]:.2f}–{eval_times[-1]:.2f} meses | {len(eval_times)} puntos")

## 3. Modelo 1 — Kaplan-Meier estratificado

Kaplan-Meier no es un modelo multivariable. Para incluirlo en una tabla comparable con los modelos predictivos, se usa como **baseline estratificado** por una covariable clínica pronóstica presente en el conjunto preprocesado.

Orden de preferencia para la estratificación:

1. `Nottingham prognostic index`
2. `Tumor Stage`
3. `Lymph nodes examined positive`
4. `Tumor Size`
5. `Age at Diagnosis`

La variable se divide en bajo/alto riesgo usando la mediana calculada en el conjunto de entrenamiento.

In [ ]:
# ==================================
# Kaplan-Meier estratificado
# ==================================

def select_km_stratifier(X):
    preferred = [
        "Nottingham prognostic index",
        "Tumor Stage",
        "Lymph nodes examined positive",
        "Tumor Size",
        "Age at Diagnosis",
    ]
    for col in preferred:
        if col in X.columns and X[col].nunique(dropna=True) > 1:
            return col

    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        raise ValueError("No hay variables numéricas disponibles para estratificar Kaplan-Meier.")
    return numeric_cols[0]


km_stratifier = select_km_stratifier(X_train)
km_threshold = float(np.median(X_train[km_stratifier]))

km_group_train = (X_train[km_stratifier].to_numpy(dtype=float) >= km_threshold)
km_group_test = (X_test[km_stratifier].to_numpy(dtype=float) >= km_threshold)

print(f"Variable de estratificación KM: {km_stratifier}")
print(f"Umbral mediana train: {km_threshold:.4f}")
print(f"Train alto riesgo: {km_group_train.sum()} | bajo riesgo: {(~km_group_train).sum()}")
print(f"Test alto riesgo : {km_group_test.sum()} | bajo riesgo: {(~km_group_test).sum()}")

km_low = KaplanMeierFitter(label=f"{km_stratifier}: bajo")
km_high = KaplanMeierFitter(label=f"{km_stratifier}: alto")

km_low.fit(dur_train[~km_group_train], event_observed=evt_train[~km_group_train])
km_high.fit(dur_train[km_group_train], event_observed=evt_train[km_group_train])

plt.figure(figsize=(7, 5))
ax = km_low.plot(ci_show=True)
km_high.plot(ax=ax, ci_show=True)
plt.title(f"Kaplan-Meier estratificado por {km_stratifier}")
plt.xlabel("Tiempo de supervivencia (meses)")
plt.ylabel("Probabilidad de supervivencia")
plt.grid(alpha=0.25)
plt.show()

def km_group_survival_array(group_mask, times):
    estimates = np.zeros((len(group_mask), len(times)), dtype=float)
    for i, is_high in enumerate(group_mask):
        km_model = km_high if is_high else km_low
        estimates[i, :] = km_model.predict(times).to_numpy(dtype=float)
    return np.clip(estimates, 0.0, 1.0)

risk_km = km_group_test.astype(float)
surv_km = km_group_survival_array(km_group_test, eval_times)

## 4. Modelo 2 — Cox LASSO

Se implementa con `CoxnetSurvivalAnalysis` y `l1_ratio=1.0`, equivalente a penalización LASSO.  
La selección de `alpha` se realiza con validación cruzada interna sobre el conjunto de entrenamiento, usando C-index de Harrell como criterio.

In [ ]:
# ==============================================
# Cox LASSO: selección de alpha con CV interna
# ==============================================

def select_cox_lasso_alpha_cv(X, y, n_alphas=60, n_splits=5, random_state=42):
    """
    Ajusta una ruta Cox LASSO inicial para obtener grid de alphas.
    Después evalúa cada alpha mediante StratifiedKFold por evento.
    """
    base = CoxnetSurvivalAnalysis(
        l1_ratio=1.0,
        n_alphas=n_alphas,
        max_iter=100_000,
        tol=1e-7,
        fit_baseline_model=False,
    )
    base.fit(X, y)
    alpha_grid = np.asarray(base.alphas_, dtype=float)

    event_strata = y["event"].astype(int)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    cv_scores = pd.DataFrame(index=np.arange(n_splits), columns=alpha_grid, dtype=float)

    for fold_idx, (tr_idx, va_idx) in enumerate(cv.split(X, event_strata)):
        X_tr = X.iloc[tr_idx] if isinstance(X, pd.DataFrame) else X[tr_idx]
        X_va = X.iloc[va_idx] if isinstance(X, pd.DataFrame) else X[va_idx]
        y_tr = y[tr_idx]
        y_va = y[va_idx]

        model = CoxnetSurvivalAnalysis(
            l1_ratio=1.0,
            alphas=alpha_grid,
            max_iter=100_000,
            tol=1e-7,
            fit_baseline_model=False,
        )
        model.fit(X_tr, y_tr)

        for alpha in alpha_grid:
            try:
                risk_va = model.predict(X_va, alpha=alpha)
                cv_scores.loc[fold_idx, alpha] = harrell_cindex(y_va, risk_va)
            except Exception:
                cv_scores.loc[fold_idx, alpha] = np.nan

    mean_scores = cv_scores.mean(axis=0, skipna=True)
    best_alpha = float(mean_scores.idxmax())

    alpha_summary = pd.DataFrame({
        "alpha": mean_scores.index.astype(float),
        "c_index_cv_mean": mean_scores.values,
        "c_index_cv_std": cv_scores.std(axis=0, skipna=True).values,
    }).sort_values("c_index_cv_mean", ascending=False)

    return best_alpha, alpha_summary, base.alphas_


best_alpha, cox_alpha_summary, cox_alpha_grid = select_cox_lasso_alpha_cv(
    X_train,
    y_train,
    n_alphas=COX_N_ALPHAS,
    n_splits=COX_INNER_CV_SPLITS,
    random_state=RANDOM_STATE,
)

print(f"Mejor alpha Cox LASSO: {best_alpha:.6g}")
display(cox_alpha_summary.head(10))

In [ ]:
# ===========================
# Ajuste final Cox LASSO
# ===========================

cox_lasso = CoxnetSurvivalAnalysis(
    l1_ratio=1.0,
    alphas=[best_alpha],
    max_iter=100_000,
    tol=1e-7,
    fit_baseline_model=True,
)
cox_lasso.fit(X_train, y_train)

risk_cox = cox_lasso.predict(X_test, alpha=best_alpha)
surv_funcs_cox = cox_lasso.predict_survival_function(X_test, alpha=best_alpha)
surv_cox = eval_step_functions(surv_funcs_cox, eval_times)

coef = pd.Series(cox_lasso.coef_[:, 0], index=feature_names, name="coef")
coef_nonzero = coef[coef.abs() > 1e-8].sort_values(key=lambda s: s.abs(), ascending=False)

print(f"Número de covariables seleccionadas por LASSO: {len(coef_nonzero)} / {len(feature_names)}")
display(coef_nonzero.head(20).to_frame())

plt.figure(figsize=(8, 6))
coef_nonzero.head(20).sort_values().plot(kind="barh")
plt.title("Cox LASSO — Top 20 coeficientes no nulos")
plt.xlabel("Coeficiente")
plt.grid(alpha=0.25)
plt.show()

## 5. Modelo 3 — Random Survival Forest

Random Survival Forest permite capturar no linealidades e interacciones sin asumir riesgos proporcionales.  
Se usa una configuración robusta y relativamente conservadora para reducir sobreajuste.

In [ ]:
# ================================
# Random Survival Forest
# ================================

rsf = RandomSurvivalForest(
    n_estimators=RSF_N_ESTIMATORS,
    min_samples_split=RSF_MIN_SAMPLES_SPLIT,
    min_samples_leaf=RSF_MIN_SAMPLES_LEAF,
    max_features=RSF_MAX_FEATURES,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

rsf.fit(X_train, y_train)

risk_rsf = rsf.predict(X_test)
surv_funcs_rsf = rsf.predict_survival_function(X_test)
surv_rsf = eval_step_functions(surv_funcs_rsf, eval_times)

print("✓ RSF entrenado")
print(f"Rango de risk score test: {np.min(risk_rsf):.3f} – {np.max(risk_rsf):.3f}")

### Importancia de variables en RSF mediante permutación *(opcional)*

`RandomSurvivalForest` de `scikit-survival` no siempre expone importancia nativa de variables.  
La alternativa recomendada es la **permutation importance** usando C-index como métrica.  
Esta celda puede tardar más; puedes reducir `n_repeats`.

In [ ]:
RUN_RSF_PERMUTATION_IMPORTANCE = False

def rsf_cindex_scorer(estimator, X, y):
    risk = estimator.predict(X)
    return harrell_cindex(y, risk)

if RUN_RSF_PERMUTATION_IMPORTANCE:
    rsf_perm = permutation_importance(
        rsf,
        X_test,
        y_test,
        scoring=rsf_cindex_scorer,
        n_repeats=5,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    rsf_importance = (
        pd.DataFrame({
            "feature": feature_names,
            "importance_mean": rsf_perm.importances_mean,
            "importance_std": rsf_perm.importances_std,
        })
        .sort_values("importance_mean", ascending=False)
    )
    display(rsf_importance.head(20))

    plt.figure(figsize=(8, 6))
    top_imp = rsf_importance.head(20).sort_values("importance_mean")
    plt.barh(top_imp["feature"], top_imp["importance_mean"])
    plt.xlabel("Pérdida media de C-index al permutar")
    plt.title("RSF — Importancia por permutación")
    plt.grid(alpha=0.25)
    plt.show()

## 6. Modelo 4 — DeepSurv

DeepSurv se implementa como una red neuronal feedforward que optimiza la pérdida parcial de Cox.  
Se separa una validación interna desde `train` para aplicar early stopping.

In [ ]:
# ================================
# DeepSurv con pycox / PyTorch
# ================================

# Split train/validación interna para early stopping
idx_all = np.arange(len(X_train_np))
idx_deep_train, idx_deep_val = train_test_split(
    idx_all,
    test_size=TEST_SIZE_DEEPSURV_VAL,
    random_state=RANDOM_STATE,
    stratify=evt_train.astype(int),
)

x_deep_train = X_train_np[idx_deep_train].astype("float32")
x_deep_val = X_train_np[idx_deep_val].astype("float32")

dur_deep_train = dur_train[idx_deep_train].astype("float32")
dur_deep_val = dur_train[idx_deep_val].astype("float32")

evt_deep_train = evt_train[idx_deep_train].astype("float32")
evt_deep_val = evt_train[idx_deep_val].astype("float32")

in_features = X_train_np.shape[1]
net = tt.practical.MLPVanilla(
    in_features=in_features,
    num_nodes=DEEPSURV_NODES,
    out_features=1,
    batch_norm=True,
    dropout=DEEPSURV_DROPOUT,
    output_bias=False,
)

deepsurv = DeepSurvModel(net, tt.optim.Adam)
deepsurv.optimizer.set_lr(DEEPSURV_LR)

callbacks = [tt.callbacks.EarlyStopping(patience=DEEPSURV_PATIENCE)]

log = deepsurv.fit(
    x_deep_train,
    (dur_deep_train, evt_deep_train),
    batch_size=DEEPSURV_BATCH_SIZE,
    epochs=DEEPSURV_EPOCHS,
    callbacks=callbacks,
    val_data=(x_deep_val, (dur_deep_val, evt_deep_val)),
    verbose=False,
)

# Necesario para obtener curvas de supervivencia
_ = deepsurv.compute_baseline_hazards()

risk_deepsurv = deepsurv.predict(X_test_np.astype("float32")).reshape(-1)
surv_df_deepsurv = deepsurv.predict_surv_df(X_test_np.astype("float32"))
surv_deepsurv = pycox_surv_df_to_array(surv_df_deepsurv, eval_times)

print("✓ DeepSurv entrenado")
print(f"Épocas ejecutadas: {len(log.to_pandas())}")
display(log.to_pandas().tail())

plt.figure(figsize=(7, 4))
log.to_pandas().plot(ax=plt.gca())
plt.title("DeepSurv — evolución de la pérdida")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.grid(alpha=0.25)
plt.show()

## 7. Comparación final en conjunto test

La tabla siguiente resume los cuatro modelos con las mismas métricas y la misma partición test.

In [ ]:
# ===============================
# Registro común de resultados
# ===============================

results = []

km_high_risk = register_result(
    results,
    model_name=f"Kaplan-Meier estratificado ({km_stratifier})",
    risk_scores=risk_km,
    survival_estimates=surv_km,
    eval_times=eval_times,
    notes="Baseline no multivariable; grupos por mediana en train.",
)

cox_high_risk = register_result(
    results,
    model_name="Cox LASSO",
    risk_scores=risk_cox,
    survival_estimates=surv_cox,
    eval_times=eval_times,
    notes=f"alpha={best_alpha:.6g}; variables seleccionadas={len(coef_nonzero)}.",
)

rsf_high_risk = register_result(
    results,
    model_name="Random Survival Forest",
    risk_scores=risk_rsf,
    survival_estimates=surv_rsf,
    eval_times=eval_times,
    notes=f"n_estimators={RSF_N_ESTIMATORS}, min_leaf={RSF_MIN_SAMPLES_LEAF}.",
)

deepsurv_high_risk = register_result(
    results,
    model_name="DeepSurv",
    risk_scores=risk_deepsurv,
    survival_estimates=surv_deepsurv,
    eval_times=eval_times,
    notes=f"MLP {DEEPSURV_NODES}, dropout={DEEPSURV_DROPOUT}, epochs={len(log.to_pandas())}.",
)

results_df = (
    pd.DataFrame(results)
    .sort_values("c_index_harrell_test", ascending=False)
    .reset_index(drop=True)
)

display(results_df)

results_path = REPORT_DIR / "metabric_model_comparison_test.csv"
results_df.to_csv(results_path, index=False)
print(f"✓ Resultados guardados en: {results_path}")

In [ ]:
# ===============================
# Visualización comparativa
# ===============================

plot_df = results_df.set_index("modelo")

plt.figure(figsize=(9, 5))
plot_df["c_index_harrell_test"].sort_values().plot(kind="barh")
plt.axvline(0.5, linestyle="--", linewidth=1)
plt.title("Comparación de modelos — C-index Harrell en test")
plt.xlabel("C-index Harrell")
plt.ylabel("")
plt.grid(axis="x", alpha=0.25)
plt.show()

plt.figure(figsize=(9, 5))
plot_df["ibs_test"].sort_values(ascending=False).plot(kind="barh")
plt.title("Comparación de modelos — Integrated Brier Score en test")
plt.xlabel("IBS (menor es mejor)")
plt.ylabel("")
plt.grid(axis="x", alpha=0.25)
plt.show()

plt.figure(figsize=(9, 5))
minus_log10_p = -np.log10(plot_df["logrank_p_test"].replace(0, np.nextafter(0, 1)))
minus_log10_p.sort_values().plot(kind="barh")
plt.title("Separación de riesgo — -log10(p) del log-rank en test")
plt.xlabel("-log10(p)")
plt.ylabel("")
plt.grid(axis="x", alpha=0.25)
plt.show()

## 8. Curvas Kaplan-Meier por grupos de riesgo predicho

Estas curvas permiten comprobar visualmente si los scores de cada modelo separan pacientes de bajo y alto riesgo.

In [ ]:
plot_km_by_group(
    dur_test,
    evt_test,
    km_high_risk,
    title=f"Test — KM estratificado por {km_stratifier}",
)

plot_km_by_group(
    dur_test,
    evt_test,
    cox_high_risk,
    title="Test — grupos de riesgo según Cox LASSO",
)

plot_km_by_group(
    dur_test,
    evt_test,
    rsf_high_risk,
    title="Test — grupos de riesgo según RSF",
)

plot_km_by_group(
    dur_test,
    evt_test,
    deepsurv_high_risk,
    title="Test — grupos de riesgo según DeepSurv",
)

## 9. Interpretación automática básica

Esta celda genera una síntesis breve a partir de la tabla de resultados.  
La interpretación académica final debe completarse en la memoria considerando intervalos de confianza, estabilidad por validación cruzada y plausibilidad clínica.

In [ ]:
best_c = results_df.iloc[0]
best_ibs = results_df.sort_values("ibs_test", ascending=True).iloc[0]

print("Resumen automático:")
print(f"- Mejor discriminación por C-index Harrell: {best_c['modelo']} ({best_c['c_index_harrell_test']:.3f}).")
print(f"- Mejor calibración/discriminación conjunta por IBS: {best_ibs['modelo']} ({best_ibs['ibs_test']:.3f}).")

for _, row in results_df.iterrows():
    p = row["logrank_p_test"]
    p_txt = "no aplicable" if pd.isna(p) else f"{p:.3e}"
    print(
        f"- {row['modelo']}: "
        f"C-index={row['c_index_harrell_test']:.3f}, "
        f"IBS={row['ibs_test']:.3f}, "
        f"log-rank p={p_txt}."
    )

## 10. Validación cruzada interna opcional

La evaluación anterior usa el split 80/20 definido en el notebook de preprocesamiento.  
Para resultados finales del TFM, se recomienda complementar con validación cruzada interna.

Por coste computacional, esta sección se controla con `RUN_INTERNAL_CV`.  
Cuando `RUN_INTERNAL_CV = True`, se reentrenan los cuatro modelos en `CV_SPLITS` particiones del conjunto de entrenamiento y se evalúan sobre cada fold de validación.

> Nota: DeepSurv es el componente más lento. Para pruebas se usan menos épocas (`CV_DEEPSURV_EPOCHS`).

In [ ]:
# ======================================================
# Validación cruzada opcional de los cuatro modelos
# ======================================================

def evaluate_fold_metrics(y_tr, y_va, dur_va, evt_va, risk_va, surv_va, times_va):
    return {
        "c_index_harrell": harrell_cindex(y_va, risk_va),
        "c_index_ipcw": uno_cindex(y_tr, y_va, risk_va),
        "ibs": safe_integrated_brier_score(y_tr, y_va, surv_va, times_va),
        "logrank_p": logrank_from_risk(dur_va, evt_va, risk_va)[1],
    }


def fit_eval_km_fold(X_tr, y_tr, X_va, y_va, times_va):
    stratifier = select_km_stratifier(X_tr)
    threshold = float(np.median(X_tr[stratifier]))
    grp_tr = X_tr[stratifier].to_numpy(dtype=float) >= threshold
    grp_va = X_va[stratifier].to_numpy(dtype=float) >= threshold

    km_low_f = KaplanMeierFitter()
    km_high_f = KaplanMeierFitter()
    km_low_f.fit(y_tr["time"][~grp_tr], event_observed=y_tr["event"][~grp_tr])
    km_high_f.fit(y_tr["time"][grp_tr], event_observed=y_tr["event"][grp_tr])

    surv = np.zeros((len(grp_va), len(times_va)), dtype=float)
    for i, is_high in enumerate(grp_va):
        model = km_high_f if is_high else km_low_f
        surv[i, :] = model.predict(times_va).to_numpy(dtype=float)

    risk = grp_va.astype(float)
    return risk, surv


def fit_eval_cox_fold(X_tr, y_tr, X_va, y_va, times_va, alpha):
    model = CoxnetSurvivalAnalysis(
        l1_ratio=1.0,
        alphas=[alpha],
        max_iter=100_000,
        tol=1e-7,
        fit_baseline_model=True,
    )
    model.fit(X_tr, y_tr)
    risk = model.predict(X_va, alpha=alpha)
    surv = eval_step_functions(model.predict_survival_function(X_va, alpha=alpha), times_va)
    return risk, surv


def fit_eval_rsf_fold(X_tr, y_tr, X_va, y_va, times_va):
    model = RandomSurvivalForest(
        n_estimators=RSF_N_ESTIMATORS,
        min_samples_split=RSF_MIN_SAMPLES_SPLIT,
        min_samples_leaf=RSF_MIN_SAMPLES_LEAF,
        max_features=RSF_MAX_FEATURES,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
    model.fit(X_tr, y_tr)
    risk = model.predict(X_va)
    surv = eval_step_functions(model.predict_survival_function(X_va), times_va)
    return risk, surv


def fit_eval_deepsurv_fold(X_tr_np, y_tr, X_va_np, y_va, times_va, epochs=96):
    tr_idx, val_idx = train_test_split(
        np.arange(len(X_tr_np)),
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=y_tr["event"].astype(int),
    )

    net = tt.practical.MLPVanilla(
        in_features=X_tr_np.shape[1],
        num_nodes=DEEPSURV_NODES,
        out_features=1,
        batch_norm=True,
        dropout=DEEPSURV_DROPOUT,
        output_bias=False,
    )
    model = DeepSurvModel(net, tt.optim.Adam)
    model.optimizer.set_lr(DEEPSURV_LR)

    model.fit(
        X_tr_np[tr_idx].astype("float32"),
        (y_tr["time"][tr_idx].astype("float32"), y_tr["event"][tr_idx].astype("float32")),
        batch_size=DEEPSURV_BATCH_SIZE,
        epochs=epochs,
        callbacks=[tt.callbacks.EarlyStopping(patience=max(10, DEEPSURV_PATIENCE // 2))],
        val_data=(
            X_tr_np[val_idx].astype("float32"),
            (y_tr["time"][val_idx].astype("float32"), y_tr["event"][val_idx].astype("float32")),
        ),
        verbose=False,
    )
    _ = model.compute_baseline_hazards()
    risk = model.predict(X_va_np.astype("float32")).reshape(-1)
    surv_df = model.predict_surv_df(X_va_np.astype("float32"))
    surv = pycox_surv_df_to_array(surv_df, times_va)
    return risk, surv


if RUN_INTERNAL_CV:
    cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    cv_rows = []

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X_train, evt_train.astype(int)), start=1):
        print(f"\nFold {fold}/{CV_SPLITS}")

        X_tr = X_train.iloc[tr_idx].copy()
        X_va = X_train.iloc[va_idx].copy()
        y_tr = y_train[tr_idx]
        y_va = y_train[va_idx]
        dur_va = y_va["time"].astype(float)
        evt_va = y_va["event"].astype(bool)
        times_va = make_eval_times(y_tr, y_va, n_times=N_EVAL_TIMES)

        # KM
        risk, surv = fit_eval_km_fold(X_tr, y_tr, X_va, y_va, times_va)
        cv_rows.append({"fold": fold, "modelo": "Kaplan-Meier estratificado", **evaluate_fold_metrics(y_tr, y_va, dur_va, evt_va, risk, surv, times_va)})

        # Cox LASSO: se usa el alpha seleccionado en el train global para mantener coste razonable.
        risk, surv = fit_eval_cox_fold(X_tr, y_tr, X_va, y_va, times_va, alpha=best_alpha)
        cv_rows.append({"fold": fold, "modelo": "Cox LASSO", **evaluate_fold_metrics(y_tr, y_va, dur_va, evt_va, risk, surv, times_va)})

        # RSF
        risk, surv = fit_eval_rsf_fold(X_tr, y_tr, X_va, y_va, times_va)
        cv_rows.append({"fold": fold, "modelo": "Random Survival Forest", **evaluate_fold_metrics(y_tr, y_va, dur_va, evt_va, risk, surv, times_va)})

        # DeepSurv
        risk, surv = fit_eval_deepsurv_fold(
            X_train_np[tr_idx],
            y_tr,
            X_train_np[va_idx],
            y_va,
            times_va,
            epochs=CV_DEEPSURV_EPOCHS,
        )
        cv_rows.append({"fold": fold, "modelo": "DeepSurv", **evaluate_fold_metrics(y_tr, y_va, dur_va, evt_va, risk, surv, times_va)})

    cv_results = pd.DataFrame(cv_rows)
    display(cv_results)

    cv_summary = (
        cv_results
        .groupby("modelo")
        .agg(
            c_index_harrell_mean=("c_index_harrell", "mean"),
            c_index_harrell_std=("c_index_harrell", "std"),
            c_index_ipcw_mean=("c_index_ipcw", "mean"),
            c_index_ipcw_std=("c_index_ipcw", "std"),
            ibs_mean=("ibs", "mean"),
            ibs_std=("ibs", "std"),
            logrank_p_median=("logrank_p", "median"),
        )
        .sort_values("c_index_harrell_mean", ascending=False)
    )

    display(cv_summary)

    cv_results.to_csv(REPORT_DIR / "metabric_model_comparison_cv_folds.csv", index=False)
    cv_summary.to_csv(REPORT_DIR / "metabric_model_comparison_cv_summary.csv")
    print(f"✓ Resultados CV guardados en: {REPORT_DIR}")
else:
    print("RUN_INTERNAL_CV = False. Cambia este parámetro a True para ejecutar validación cruzada interna.")

## 11. Guardado de modelos y resultados

Se guardan los modelos entrenados y los objetos principales para reproducibilidad.  
DeepSurv se guarda como red neuronal (`.pt`) mediante `save_net`.

In [ ]:
# ===============================
# Guardado de modelos y artefactos
# ===============================

joblib.dump(km_low, MODEL_DIR / "km_low.joblib")
joblib.dump(km_high, MODEL_DIR / "km_high.joblib")
joblib.dump(
    {
        "stratifier": km_stratifier,
        "threshold": km_threshold,
    },
    MODEL_DIR / "km_metadata.joblib",
)

joblib.dump(cox_lasso, MODEL_DIR / "cox_lasso.joblib")
joblib.dump(
    {
        "best_alpha": best_alpha,
        "coef_nonzero": coef_nonzero,
        "alpha_cv_summary": cox_alpha_summary,
    },
    MODEL_DIR / "cox_lasso_metadata.joblib",
)

joblib.dump(rsf, MODEL_DIR / "rsf.joblib")

deepsurv.save_net(str(MODEL_DIR / "deepsurv_net.pt"))
joblib.dump(
    {
        "nodes": DEEPSURV_NODES,
        "dropout": DEEPSURV_DROPOUT,
        "batch_size": DEEPSURV_BATCH_SIZE,
        "epochs_executed": len(log.to_pandas()),
        "learning_rate": DEEPSURV_LR,
    },
    MODEL_DIR / "deepsurv_metadata.joblib",
)

# Scores de riesgo por paciente del test
risk_scores_test = pd.DataFrame({
    "duration": dur_test,
    "event": evt_test.astype(int),
    "risk_km": risk_km,
    "risk_cox_lasso": risk_cox,
    "risk_rsf": risk_rsf,
    "risk_deepsurv": risk_deepsurv,
})
risk_scores_test.to_csv(REPORT_DIR / "metabric_test_risk_scores.csv", index=False)

print(f"✓ Modelos guardados en: {MODEL_DIR}")
print(f"✓ Reports guardados en: {REPORT_DIR}")

## 12. Puntos para redactar en la memoria

Cuando ejecutes el notebook y obtengas resultados reales, puedes trasladar a la memoria:

- **Tabla comparativa principal**: C-index, IBS y log-rank por modelo.
- **Figura de barras del C-index**: discriminación predictiva.
- **Figura de barras del IBS**: calibración/discriminación conjunta.
- **Curvas Kaplan-Meier por riesgo alto/bajo**: utilidad clínica de la estratificación.
- **Coeficientes no nulos de Cox LASSO**: primera capa de interpretabilidad.
- **Permutation importance de RSF** *(si se activa)*: importancia no lineal de variables.
- **Limitación de KM**: no es multivariable; funciona como baseline estratificado.
- **Limitación de DeepSurv**: mayor flexibilidad, pero mayor coste computacional y menor interpretabilidad.

La interpretación final debe priorizar no solo el mejor C-index, sino también la estabilidad de CV, el IBS, la separación por log-rank y la coherencia clínica de las variables relevantes.